# Yearly simulation DC and WCT only unsconstrained

1. Setup environment
2. Loop for each day
   1. Run water distribution -> $V_{avail}$ for each day in month
   2. Run daily optimization -> decision variables for each day in month
   3. Simulate envinronment*
3. Result is a dataframe with as many rows as days in the year and as columns attributes of `OperationPoint`
4. Visualize simulation results

In [1]:
import os
MR = "/home/patomareao/MATLAB/R2024b"
os.environ['LD_LIBRARY_PATH'] = f"{MR}/runtime/glnxa64:\
{MR}/bin/glnxa64:\
{MR}/sys/os/glnxa64:\
{MR}/extern/bin/glnxa64:\
/lib/x86_64-linux-gnu:"


In [1]:
from pathlib import Path
from dataclasses import asdict
import numpy as np
import pandas as pd
# from IPython.display import SVG
from loguru import logger
import pygmo as pg
import hjson

import combined_cooler_model
from solhycool_modeling import OperationPoint

from solhycool_modeling import EnvironmentVariables
from solhycool_optimization.problems.horizon import WctRestrictedProblem as Problem
from solhycool_optimization.utils.evaluation import optimize

# Visualization packages
from solhycool_optimization.visualization import plot_obj_scape_comp_1d
# from solhycool_visualization.operation import plot_hydraulic_distribution
# from solhycool_visualization.diagrams import WascopStateVisualizer
from solhycool_optimization.visualization import visualize_solutions_distribution
from solhycool_visualization.operation import plot_results

logger.disable("phd_visualizations")

data_path: Path = Path("../../data")
env_path: Path = data_path / "datasets/environment_data_20220101_20241231.h5"
base_output_path: Path = Path("../results")
diagram_path: Path = Path("/workspaces/SOLhycool/data/assets/base_diagram.svg")
date_span: tuple[str, str] = ("20220101", "20221231")

cc_model = combined_cooler_model.initialize()
np.set_printoptions(precision=2)

%load_ext autoreload
%autoreload 2


Authorization required, but no authorization protocol specified



In [ ]:
import matlab


ModuleNotFoundError: No module named 'matlab'

## 1. Setup environment

In [3]:
output_path = base_output_path / f"evaluation_{date_span[0]}_{date_span[1]}"
if not output_path.exists():
    output_path.mkdir(parents=True)
   
# Load environment into EnvironmentVariables for the episode
df_env = pd.read_hdf(env_path).loc[date_span[0]:date_span[1]]
display(df_env)
    
env_vars = EnvironmentVariables.from_dataframe(df_env)
env_vars.Q = env_vars.Q/2 # With only one component we can only get to half of the nominal power


,G_Gh,G_Dh,G_Gk,G_Bn,Tamb_C,HR_pct,hs,Az,precip_mm,Td,...,Q_kW,Tv_C,water_price_morocco_eur_m3,water_price_morocco_alternative_eur_m3,water_price_egypt_eur_m3,water_price_egypt_alternative_eur_m3,Vavail_m3,deltaV_m3_h,Vavail_l,deltaV_l_h
2022-01-01 00:00:00+00:00,0.0,0.0,0.0,0.0,10.7,90.0,0.0,-144.0,6.7,9.1,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,19.083485,-0.916515,19083.485034,-916.514966
2022-01-01 01:00:00+00:00,0.0,0.0,0.0,0.0,11.6,82.0,0.0,-163.4,0.5,8.7,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,19.020231,-0.979769,19020.230685,-979.769315
2022-01-01 02:00:00+00:00,0.0,0.0,0.0,0.0,11.3,82.0,0.0,-124.4,0.5,8.4,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,19.002447,-0.997553,19002.447115,-997.552885
2022-01-01 03:00:00+00:00,0.0,0.0,0.0,0.0,11.2,82.0,0.0,-105.4,0.3,8.3,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,18.965136,-1.034864,18965.135822,-1034.864178
2022-01-01 04:00:00+00:00,0.0,0.0,0.0,0.0,11.0,82.0,0.0,-93.6,0.2,8.1,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,18.968172,-1.031828,18968.171943,-1031.828057
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-12-31 19:00:00+00:00,0.0,0.0,0.0,0.0,9.1,83.0,0.0,73.4,0.2,6.4,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,19.024184,-0.975816,19024.183721,-975.816279
2022-12-31 20:00:00+00:00,0.0,0.0,0.0,0.0,8.4,87.0,0.0,81.4,0.2,6.4,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,19.065815,-0.934185,19065.815492,-934.184508
2022-12-31 21:00:00+00:00,0.0,0.0,0.0,0.0,7.7,93.0,0.0,90.0,0.1,6.6,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,18.922617,-1.077383,18922.616810,-1077.383190
2022-12-31 22:00:00+00:00,0.0,0.0,0.0,0.0,7.0,96.0,0.0,100.2,0.3,6.3,...,200.0,35.0,0.029054,0.581078,0.060471,1.209412,18.962058,-1.037942,18962.058138,-1037.941862


In [4]:
from datetime import datetime

date_span: tuple[str, str] = ("20220101", "20221231")

start_date = datetime.strptime(date_span[0], "%Y%m%d").replace(hour=0, minute=0, second=0)
end_date = datetime.strptime(date_span[1], "%Y%m%d").replace(hour=23, minute=0, second=0)

start_date.month


1